# 01 - Problem Formulation: Max-Cut and Ising Hamiltonian

This notebook explains the mathematical formulation of the Max-Cut problem and its conversion to a quantum Hamiltonian (Ising model).

## Table of Contents
1. [Max-Cut Problem Definition](#max-cut)
2. [Classical Formulation](#classical)
3. [Ising Model Mapping](#ising)
4. [Hamiltonian Construction](#hamiltonian)
5. [Graph Generation](#graph)

<a id='max-cut'></a>
## 1. Max-Cut Problem Definition

The **Max-Cut** problem is a classic combinatorial optimization problem:

Given an undirected graph $G = (V, E)$:
- **Goal**: Partition vertices $V$ into two disjoint sets to maximize the number of edges crossing between them
- **Application**: Network design, VLSI, clustering, robot communication

### Example

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Create a simple example graph
G = nx.Graph()
G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0), (0, 2)])

plt.figure(figsize=(8, 4))

# Draw the graph
plt.subplot(1, 2, 1)
nx.draw(G, with_labels=True, node_color='lightblue', node_size=500)
plt.title('Original Graph')
plt.axis('off')

# Show a cut
plt.subplot(1, 2, 2)
partition = {0, 1}  # Partition A: nodes 0, 1
pos = nx.spring_layout(G)

# Color by partition
node_colors = ['red' if n in partition else 'blue' for n in G.nodes()]
nx.draw(G, pos, with_labels=True, node_color=node_colors, node_size=500)

# Highlight cut edges
cut_edges = [(u, v) for u, v in G.edges() if (u in partition) != (v in partition)]
nx.draw_networkx_edges(G, pos, edgelist=cut_edges, edge_color='green', width=3)

plt.title('Max-Cut Solution\n(Green edges = cut edges)')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Cut edges: {cut_edges}")
print(f"Cut value: {len(cut_edges)} edges")

<a id='classical'></a>
## 2. Classical Formulation

### Binary Variable Representation

Let $x_i \in \{0, 1\}$ denote which partition vertex $i$ belongs to:
- $x_i = 0$: vertex $i$ in partition A
- $x_i = 1$: vertex $i$ in partition B

### Cost Function

For edge $(i, j)$, it contributes to the cut if $x_i \neq x_j$:

$$C_{ij} = x_i + x_j - 2 x_i x_j = x_i (1 - x_j) + x_j (1 - x_i)$$

Or equivalently:

$$C_{ij} = x_i + x_j - 2 x_i x_j$$

### Total Cost

$$C = \sum_{(i,j) \in E} (x_i + x_j - 2 x_i x_j)$$

<a id='ising'></a>
## 3. Ising Model Mapping

### Transformation to Ising Variables

Convert binary variables to spin variables:

$$z_i = 2x_i - 1$$

This maps:
- $x_i = 0 \rightarrow z_i = -1$
- $x_i = 1 \rightarrow z_i = +1$

### Ising Cost Function

The edge contribution becomes:

$$C_{ij} = \frac{1}{2}(1 - z_i z_j)$$

Proof:
- If $z_i = z_j$ (same partition): $C_{ij} = (1 - 1)/2 = 0$
- If $z_i \neq z_j$ (different partitions): $C_{ij} = (1 - (-1))/2 = 1$

### Total Hamiltonian

$$H = \sum_{(i,j) \in E} \frac{1}{2}(1 - Z_i Z_j)$$

### LaTeX Derivation

$$\begin{aligned}
C &= \frac{1}{2} \sum_{(i,j) \in E} (1 - Z_i Z_j) \\\\
  &= \frac{1}{2} \sum_{(i,j) \in E} 1 - \frac{1}{2} \sum_{(i,j) \in E} Z_i Z_j \\\\
  &= \text{constant} - \frac{1}{2} \sum_{(i,j) \in E} Z_i Z_j
\end{aligned}$$

The **ground state** (lowest energy) of this Hamiltonian corresponds to the **optimal Max-Cut solution**.

<a id='hamiltonian'></a>
## 4. Hamiltonian Construction

In [ ]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

def build_maxcut_hamiltonian(graph):
    """
    Build Ising Hamiltonian for Max-Cut.
    
    H = Σ w_ij * (1 - Z_i Z_j) / 2
    """
    n_qubits = graph.number_of_nodes()
    edges = list(graph.edges())
    
    # Get edge weights
    if nx.is_weighted(graph):
        weights = nx.get_edge_attributes(graph, 'weight')
    else:
        weights = {edge: 1.0 for edge in edges}
    
    pauli_list = []
    coeffs = []
    
    for i, j in edges:
        w = weights.get((i, j), weights.get((j, i), 1.0))
        
        # Z_i Z_j term: -w/2 * Z_i Z_j
        pauli_str = ['I'] * n_qubits
        pauli_str[i] = 'Z'
        pauli_str[j] = 'Z'
        pauli_str = ''.join(reversed(pauli_str))  # Qiskit uses MSB first
        
        pauli_list.append(pauli_str)
        coeffs.append(-w / 2)
    
    # Constant offset
    offset = sum(weights.values()) / 2
    
    # Create Hamiltonian
    hamiltonian = SparsePauliOp(pauli_list, coeffs=coeffs)
    hamiltonian = hamiltonian.simplify()
    
    return hamiltonian, offset

# Build Hamiltonian for our example graph
hamiltonian, offset = build_maxcut_hamiltonian(G)

print("Hamiltonian Terms:")
print(f"  Number of terms: {len(hamiltonian)}")
print(f"  Offset: {offset}")
print(f"\nHamiltonian:")
print(hamiltonian)

### Pauli Operator Explanation

The Hamiltonian consists of $Z_i Z_j$ terms:

| Term | Meaning |
|------|---------|
| $Z_i$ | Pauli-Z on qubit i |
| $Z_i Z_j$ | ZZ interaction between qubits i and j |
| Coefficient | Weight of the edge |

For the example graph with edges (0,1), (1,2), (2,3), (3,0), (0,2):

$$H = -\frac{1}{2}Z_0 Z_1 - \frac{1}{2}Z_1 Z_2 - \frac{1}{2}Z_2 Z_3 - \frac{1}{2}Z_3 Z_0 - \frac{1}{2}Z_0 Z_2 + 2.5 \cdot I$$

<a id='graph'></a>
## 5. Graph Generation for Robot Networks

In [ ]:
import sys
sys.path.append('..')

from src.graph_generator import GraphGenerator

# Create generator
gen = GraphGenerator(seed=42)

# Generate robot mesh network (D-regular graph)
robot_mesh = gen.generate_robot_mesh(n_robots=15, connectivity=3, seed=42)

print(f"Robot Communication Mesh:")
print(f"  Nodes (robots): {robot_mesh.number_of_nodes()}")
print(f"  Edges (links): {robot_mesh.number_of_edges()}")
print(f"  Average degree: {2 * robot_mesh.number_of_edges() / robot_mesh.number_of_nodes():.2f}")

# Visualize
plt.figure(figsize=(10, 6))
pos = nx.spring_layout(robot_mesh, seed=42)
nx.draw(robot_mesh, pos, with_labels=True, node_color='lightgreen', 
        node_size=200, edge_color='gray', alpha=0.8)
plt.title('Robot Communication Mesh Network\n(15 robots, 3 connections each)')
plt.show()

In [ ]:
# Build Hamiltonian for robot mesh
hamiltonian_robot, offset_robot = build_maxcut_hamiltonian(robot_mesh)

print(f"Hamiltonian for Robot Mesh:")
print(f"  Number of terms: {len(hamiltonian_robot)}")
print(f"  Offset: {offset_robot}")
print(f"\nHamiltonian (first 10 terms):")
for i, (pauli, coeff) in enumerate(zip(hamiltonian_robot.paulis[:10], hamiltonian_robot.coeffs[:10])):
    print(f"  {coeff:.2f} * {pauli.to_label()}")
if len(hamiltonian_robot) > 10:
    print(f"  ... and {len(hamiltonian_robot) - 10} more terms")

## Summary

In this notebook, we covered:

1. **Max-Cut Problem**: Partition graph vertices to maximize crossing edges
2. **Binary Variables**: $x_i \in \{0, 1\}$ for partition assignment
3. **Ising Mapping**: $z_i = 2x_i - 1$ transforms to spin variables
4. **Hamiltonian**: $H = \sum_{(i,j)} \frac{1}{2}(1 - Z_i Z_j)$
5. **Graph Generation**: D-regular graphs for robot communication networks

In the next notebook, we'll implement the QAOA circuit to solve this problem!